# سبق 13 - ایجنٹ میموری کے ساتھ کوگنی نالج گرافز


## ترتیب

یہ نوٹ بُک اس بات کا مظاہرہ کرتی ہے کہ کس طرح ایک ذہین **کوڈنگ اسسٹنٹ** بنایا جا سکتا ہے جو مستقل یادداشت کے ساتھ [**Cognee**](https://www.cognee.ai/) نالج گراف اور **Microsoft Agent Framework** (MAF) کا استعمال کرتا ہو۔

Cognee غیر منظم متن کو ایک منظم، قابلِ استفسار نالج گراف میں تبدیل کرتا ہے جو ویکٹر ایمبیڈنگز کی حمایت کرتا ہے — تاکہ آپ کے ایجنٹ کو ایک وسیع، تعلقات سے آگاہ طویل مدتی یادداشت ملے۔

### آپ کیا سیکھیں گے
1. **نالج گراف بنانا**: ڈویلپر پروفائلز اور بہترین طریقوں کو منظم، قابلِ استفسار علم میں تبدیل کرنا۔
2. **Cognee کو MAF کے ساتھ مربوط کرنا**: `@tool` فنکشنز کا استعمال کرتے ہوئے MAF ایجنٹ کو Cognee کے نالج گراف سے استفسار کرنے دینا۔
3. **سیشن-آگاہ بات چیت**: ایک ہی سیشن میں متعدد سوالات کے دوران سیاق و سباق کو برقرار رکھنا۔
4. **طویل مدتی یادداشت**: اہم معلومات کو سیشنز کے پار برقرار رکھنا اور نئی بات چیت میں اسے بازیافت کرنا۔

### پیشگی ضروریات
- پائتھن 3.9+
- لوکل ریڈیس چل رہا ہو (`docker run -d -p 6379:6379 redis`) سیشن مینیجمنٹ کے لیے
- ایک LLM API کلید (مثلاً OpenAI) — `.env` میں `LLM_API_KEY` سیٹ کریں
- `.env` میں `CACHING=true` (Cognee سیشنز کے لیے ضروری)
- Azure AI Foundry پروجیکٹ جس میں چیٹ ماڈل تعینات ہو
- `.env` میں `AZURE_AI_PROJECT_ENDPOINT` اور `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI میں تصدیق شدہ لاگ ان (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ایجنٹ میموری کی اقسام

یہ نوٹ بک میموری کی وہی تین اقسام کو دیکھتی ہے جو مین لیسن 13 نوٹ بک میں تھیں، لیکن لانگ ٹرم میموری کے لیے کوگنی کو بطور بیک اینڈ استعمال کرتی ہے:

| میموری کی قسم | میکانزم | زندگی کا دورانیہ |
|---|---|---|
| **ورکنگ** | `agent.create_session()` (MAF) | ایک ہی گفتگو |
| **شارٹ ٹرم** | کوگنی سیشن کیش (Redis) | ایک ہی سیشن |
| **لانگ ٹرم** | کوگنی نالج گراف + ویکٹرز | مستقل |

### کوگنی کی میموری کی ساخت
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## کوگنی اسٹوریج کی تیاری کریں


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## حصہ 1 — علم کا ذخیرہ بنانا

ہم اپنے کوڈنگ اسسٹنٹ کے لیے ایک جامع علم کا ذخیرہ بنانے کے لیے تین اقسام کے ڈیٹا کو شامل کرتے ہیں:

1. **ڈیولپر پروفائل** — ذاتی مہارت اور تکنیکی پس منظر  
2. **پائتھن بہترین طریقے** — پائتھن کا زین عملی ہدایات کے ساتھ  
3. **تاریخی گفتگوئیں** — ماضی کے سوال و جواب کے سیشنز جو ڈیولپرز اور AI اسسٹنٹس کے درمیان ہوئے


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## علم کا گراف بصری بنائیں

Cognee ان اداروں اور تعلقات کی ایک تعاملی HTML بصری تصویر پیش کر سکتا ہے جو اس نے نکالی ہے۔


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## یادداشت کو Memify کے ساتھ مالا مال کریں

`memify()` نالج گراف کا تجزیہ کرتا ہے اور ذہین قواعد تیار کرتا ہے — جو پیٹرنز، بہترین طریقے، اور تصورات کے درمیان تعلقات کی شناخت کرتا ہے۔


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## حصہ 2 — MAF ایجنٹ Cognee ٹولز کے ساتھ

اب ہم ایک MAF ایجنٹ بناتے ہیں جو `@tool` فنکشنز کے ذریعے Cognee کے نالج گراف سے سوالات کر سکتا ہے۔ اس سے ایجنٹ کو گراف سے آگاہ سیمنٹک سرچ کی مکمل طاقت استعمال کرنے کی اجازت ملتی ہے جبکہ سیشنز کے ذریعے مکالماتی سیاق و سباق کو برقرار رکھا جاتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## ورکنگ میموری بمطابق سیشنز

`AgentSession` (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) سیشن کے اندر ورکنگ میموری فراہم کرتا ہے۔ ایجنٹ پہلے کے پیغامات کی طرف رجوع کر سکتا ہے جبکہ ساتھ ہی Cognee کے طویل مدتی علم کے گراف سے بھی سوالات کر سکتا ہے۔


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## نئی سیشن — طویل مدتی یادداشت برقرار رہتی ہے

نئی سیشن شروع کرنے سے ورکنگ میموری صاف ہو جاتی ہے، لیکن Cognee نالج گراف اب بھی دستیاب ہے۔ ایجنٹ بالکل نئے گفتگو میں بھی وہی طویل مدتی معلومات حاصل کر سکتا ہے۔


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصہ

اس نوٹ بک میں آپ نے ایک کوڈنگ اسسٹنٹ بنایا جو **MAF کی ورکنگ میموری** (`agent.create_session()`) کو **Cognee کے طویل مدتی نالج گراف** کے ساتھ ملاتی ہے۔

### آپ نے کیا سیکھا
1. **نالج گراف کی تعمیر**: Cognee غیر ساختہ متن کو استعمال کرتا ہے اور ایک گراف + ویکٹر میموری بناتا ہے۔
2. **memify کے ساتھ گراف کی بہتری**: آپ کے موجودہ گراف پر مشتق حقائق اور زیادہ گہرے تعلقات شامل کرنا۔
3. **MAF + Cognee انضمام**: `@tool` فنکشنز ایک MAF ایجنٹ کو Cognee کے گراف سے قدرتی انداز میں سوال کرنے دیتے ہیں۔
4. **ورکنگ میموری + طویل مدتی میموری**: `AgentSession` (جو `agent.create_session()` کے ذریعے حاصل ہوتی ہے) سیشن کا سیاق و سباق فراہم کرتی ہے جب کہ Cognee مستقل علم مہیا کرتا ہے۔
5. **NodeSets کے ساتھ فلٹر شدہ تلاش**: نالج گراف کے مخصوص ذیلی حصوں کو ہدف بنانا (مثلاً صرف اصول)۔

### اہم نکات
- **Cognee** خام متن کو ساختہ، تعلقات پر مبنی یادداشت میں تبدیل کرتا ہے — جو ایک سادہ ویکٹر اسٹور سے زیادہ طاقتور ہے۔
- **`@tool` فنکشنز** MAF ایجنٹس اور بیرونی نالج سسٹمز کے درمیان واضح رابطہ فراہم کرتے ہیں۔
- **`AgentSession`** (جو `agent.create_session()` کے ذریعے دستیاب ہے) ہر گفتگو کے سیاق کو طویل المدتی علم سے جدا رکھتا ہے۔
- وہی نالج گراف متعدد سیشنز اور ایجنٹس کی خدمت کرتا ہے۔

### حقیقی دنیا کی درخواستیں
- **ڈیولپر کوپائلٹس**: کوڈ ریویو، حادثاتی تجزیہ، فن تعمیر کی معاونت
- **کسٹمر-مخاطب کوپائلٹس**: مصنوعی دستاویزات، FAQs، اور CRM نوٹس پر مبنی سپورٹ ایجنٹس
- **اندرونی ماہر کوپائلٹس**: پالیسیاں، قانونی، یا سیکورٹی معاونین جو رہنما خطوط پر غور کرتے ہیں
- **متحد ڈیٹا پرتیں**: ساختہ اور غیر ساختہ ڈیٹا کو ایک قابل تلاش گراف میں ملانا

### اگلے اقدامات
- Cognee میں وقتی آگاہی پر تجربات کریں
- مخصوص ڈومین کے لیے OWL اونٹولوجی کی تعریف کریں تاکہ گراف کی کوالٹی بہتر ہو
- وقت کے ساتھ بازیابی کو بہتر بنانے کے لیے صارف کی رائے کے لوپ شامل کریں
- ایک ہی Cognee میموری پرت شیئر کرنے والے کثیر ایجنٹ نظام کو وسعت دیں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈسکلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا مغالطے ہوسکتے ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھی جائے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والے کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم پر نہیں ہوتی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
